In [28]:
# Initial setup

import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.0f}".format

In [29]:
# The data cleaning process in the previous step.
# And data pipeline from the previous step
# We put them into a single function.

In [30]:
def ETL_pipeline(year:int):
    df = pd.read_csv(f"./data/raw/generation_tech_{year}.csv")
    # Clean up generation values
    df['Generation (MW)'] = pd.to_numeric(df['Generation (MW)'], errors='coerce')
    # Extract date from MTU
    df['start_time'] = df['MTU (CET/CEST)'].str.split(' - ').str[0]
    df['date'] = df['start_time'].str.split(' ').str[0]
    df['date'] = pd.to_datetime(df['date'], format='%d/%m/%Y')
    df = df[['date','Production Type','Generation (MW)']]
    df = df.rename(columns={
        'Production Type':'tech',
        'Generation (MW)':'gen',
    })
    df['daily_electricity_by_gen'] = (
        df.groupby(['date', 'tech'])['gen']
        .transform('sum')
    )
    df['daily_electricity'] = (
        df.groupby(['date'])['gen']
        .transform('sum')
    )
    df['daily_avg_share_gen'] = df['daily_electricity_by_gen'] / df['daily_electricity']
    g = df.groupby(['date','tech']).agg(
        avg_gen=('gen','mean'),
        std_gen=('gen','std'),
    ).reset_index()
    g['cov'] = g['std_gen'] / g['avg_gen'] # This will be a percentage in Power BI
    g2 = df.groupby(['date','tech']).agg(
        daily_electricity_by_gen=('daily_electricity_by_gen','mean'),
        daily_avg_share_gen=('daily_avg_share_gen','mean')
    ).reset_index()
    m = pd.merge(g,g2,'left',['date','tech'])
    g3 = df.groupby(['date']).agg(daily_electricity=('daily_electricity','mean')).reset_index()
    
    return m,g3

In [31]:
m,g3 = ETL_pipeline(year=2015)

In [32]:
m.head()

,date,tech,avg_gen,std_gen,cov,daily_electricity_by_gen,daily_avg_share_gen
0,2015-01-01,Biomass,201,6,0,"4,813",0
1,2015-01-01,Energy storage,NaN,NaN,NaN,0,0
2,2015-01-01,Fossil Brown coal/Lignite,NaN,NaN,NaN,0,0
3,2015-01-01,Fossil Coal-derived gas,NaN,NaN,NaN,0,0
4,2015-01-01,Fossil Gas,"2,425",148,0,"58,190",0


In [33]:
g3.head()

,date,daily_electricity
0,2015-01-01,"1,725,680"
1,2015-01-02,"1,793,290"
2,2015-01-03,"1,747,153"
3,2015-01-04,"1,643,712"
4,2015-01-05,"1,841,655"


In [34]:
# Loop over all raw datasets from 2015 - 2025
# Create two "compilations"

In [35]:
daily_electricity = pd.DataFrame()
tech_stats = pd.DataFrame()

for year in range(2015,2026):
    m,g3 = ETL_pipeline(year)
    daily_electricity = pd.concat([daily_electricity,g3])
    tech_stats = pd.concat([tech_stats,m])

In [36]:
daily_electricity

,date,daily_electricity
0,2015-01-01,"1,725,680"
1,2015-01-02,"1,793,290"
2,2015-01-03,"1,747,153"
3,2015-01-04,"1,643,712"
4,2015-01-05,"1,841,655"
...,...,...
360,2025-12-27,"7,201,406"
361,2025-12-28,"7,299,374"
362,2025-12-29,"7,504,249"
363,2025-12-30,"7,678,297"


In [37]:
tech_stats

,date,tech,avg_gen,std_gen,cov,daily_electricity_by_gen,daily_avg_share_gen
0,2015-01-01,Biomass,201,6,0,"4,813",0
1,2015-01-01,Energy storage,NaN,NaN,NaN,0,0
2,2015-01-01,Fossil Brown coal/Lignite,NaN,NaN,NaN,0,0
3,2015-01-01,Fossil Coal-derived gas,NaN,NaN,NaN,0,0
4,2015-01-01,Fossil Gas,"2,425",148,0,"58,190",0
...,...,...,...,...,...,...,...
7660,2025-12-31,Other renewable,NaN,NaN,NaN,0,0
7661,2025-12-31,Solar,"2,309","3,815",2,"221,687",0
7662,2025-12-31,Waste,353,4,0,"33,876",0
7663,2025-12-31,Wind Offshore,740,185,0,"71,015",0


In [38]:
tech_stats.to_csv('./data/tech_stats.csv', index=False)

In [39]:
daily_electricity.to_csv('./data/daily_electricity.csv', index=False)